# Submissao 1 - DNN
Notebook limpo para inferencia final: le CSV, preprocessa, carrega modelo e guarda submissao.

In [ ]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.data_processing import clean_text
from src.bag_of_words import NumpyBagOfWords
from src.models_numpy.dnn.neuralnet import NeuralNetwork
from src.models_numpy.dnn.layers import DenseLayer
from src.models_numpy.dnn.activation import ReLUActivation, SoftmaxActivation

input_csv = Path('../submissions/submissao1/subm1.csv')
train_csv = Path('../data/processed/dataset_kaggle_daigt_processed.csv')
model_path = Path('../saved_models/DNN_final.pkl')
bow_path = Path('../saved_models/DNN_bow_model.pkl')
output_dir = Path('../submissions/submissao1/results')
output_csv = output_dir / 'submission_dnn.csv'

MAX_WORDS = 2500
HIDDEN_NEURONS = 164
NUM_CLASSES = 5

for required in [input_csv, train_csv, model_path, bow_path]:
    if not required.exists():
        raise FileNotFoundError(f'Ficheiro em falta: {required}')

output_dir.mkdir(parents=True, exist_ok=True)

df_train = pd.read_csv(train_csv, sep=';')
label_map = {label: i for i, label in enumerate(df_train['Label'].unique())}
idx_to_label = {i: label for label, i in label_map.items()}

df_sub = pd.read_csv(input_csv, sep=';')
df_sub['text_clean'] = df_sub['Text'].fillna('').apply(clean_text)

bow = NumpyBagOfWords(max_words=MAX_WORDS)
bow.load(str(bow_path))
X_sub = bow.transform(df_sub['text_clean'])

model = NeuralNetwork()
model.add(DenseLayer(input_size=MAX_WORDS, output_size=HIDDEN_NEURONS))
model.add(ReLUActivation())
model.add(DenseLayer(input_size=HIDDEN_NEURONS, output_size=NUM_CLASSES))
model.add(SoftmaxActivation())
model.load(str(model_path))

pred_ids = model.predict(X_sub)

df_sub['Label'] = [idx_to_label[int(i)] for i in pred_ids]
df_out = df_sub[['ID', 'Label']]
df_out.to_csv(output_csv, sep=';', index=False)

print(f'Submissao DNN guardada em: {output_csv}')
print(df_out.head())

Submissao DNN guardada em: ../submissions/submissao1/results/submission_dnn.csv
     ID   Label
0  D2-1  OpenAI
1  D2-2  OpenAI
2  D2-3    Meta
3  D2-4  OpenAI
4  D2-5  OpenAI
